In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_provider_stats_table = "health.default.silver_provider_statistics"
silver_provider_diagnosis_patterns_table = "health.default.silver_provider_diagnosis_patterns"
bronze_fraud_labels_table = "health.default.bronze_fraud_labels"
gold_provider_fraud_risk_table = "health.default.gold_provider_fraud_risk"

# Combine all provider features
df_provider_complete = spark.table(silver_provider_stats_table) \
    .join(
        spark.table(silver_provider_diagnosis_patterns_table),
        "Provider",
        "left"
    )

# Calculate composite fraud risk score
df_provider_risk = df_provider_complete \
    .withColumn("risk_score",
                (F.col("reimbursement_zscore") * 0.3) +
                (F.col("cpp_zscore") * 0.2) +
                (F.when(F.col("diagnosis_diversity_ratio") < 0.1, 5).otherwise(0)) +
                (F.when(F.col("claims_per_patient") > 20, 10).otherwise(0)) +
                (F.when(F.col("avg_claim_amount") > 10000, 5).otherwise(0))
    ) \
    .withColumn("risk_percentile", 
                F.percent_rank().over(Window.orderBy("risk_score")))

# Join with ground truth labels
df_provider_labeled = df_provider_risk \
    .join(
        spark.table(bronze_fraud_labels_table),
        "Provider",
        "left"
    ) \
    .fillna(0, subset=["is_fraud"])

df_provider_labeled.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_provider_fraud_risk_table)

print(f"Created {gold_provider_fraud_risk_table}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_beneficiary_profiles_table = "health.default.silver_beneficiary_profiles"
gold_patient_cohorts_table = "health.default.gold_patient_cohorts"

# Create patient cohorts for population health management
df_cohorts = spark.table(silver_beneficiary_profiles_table) \
    .select(
        "BeneID",
        "age",
        "Gender",
        "State",
        "chronic_condition_count",
        "risk_category",
        "total_costs",
        "inpatient_visits",
        "unique_providers"
    )

# Cohort definitions
df_cohorts = df_cohorts \
    .withColumn("cohort",
                F.when(F.col("chronic_condition_count") >= 3, "High Complexity")
                 .when(F.col("inpatient_visits") >= 2, "Frequent Admissions")
                 .when(F.col("age") >= 75, "Elderly")
                 .otherwise("Standard"))

# Aggregate by cohort
df_cohort_summary = df_cohorts \
    .groupBy("cohort") \
    .agg(
        F.count("*").alias("patient_count"),
        F.avg("total_costs").alias("avg_costs_per_patient"),
        F.sum("total_costs").alias("total_cohort_costs"),
        F.avg("inpatient_visits").alias("avg_inpatient_visits"),
        F.avg("chronic_condition_count").alias("avg_chronic_conditions")
    ) \
    .withColumn("cost_per_patient_rank", 
                F.dense_rank().over(Window.orderBy(F.desc("avg_costs_per_patient"))))

df_cohort_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_patient_cohorts_table)

print(f"Created {gold_patient_cohorts_table}")